<img src="https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@bf9431e/ressources/img/logo_macmia.png" alt="Banque des Territoires · France 2030 · MACMIA" width="520">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees_v2/corrections/seance1_correction.ipynb)

# Séance 2.1 — Charger, comprendre et nettoyer une base de données

**Correction** · durée : 6h (2h de cours, 4h d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- charger un fichier de données depuis le web en une ligne
- décrire un fichier que vous n'avez jamais vu en 30 secondes
- sélectionner exactement les lignes et les colonnes qui vous intéressent
- calculer des indicateurs simples sur une colonne entière
- lire un message d'erreur au lieu de le subir

## Le contexte

Vous venez d'arriver chez un **détaillant en ligne** européen. On vous remet
l'historique des ventes de l'année écoulée et une question simple :

> *« Sur quel marché faut-il investir l'an prochain ? »*

Vous ne pouvez pas répondre tant que vous ne savez pas ce que contient ce
fichier. Cette séance, c'est exactement ça : **prendre en main un jeu de
données qu'on n'a jamais vu**.

Nous avons trois fichiers :

| Fichier | Une ligne = | Colonnes |
|---|---|---|
| `ventes.csv` | un produit dans une commande | `date`, `cmd_id`, `prod_id`, `qte`, `prix`, `client_id` |
| `clients.csv` | un client | `client_id`, `pays`, `segment`, `date_insc` |
| `produits.csv` | un produit | `prod_id`, `libelle`, `categorie` |

> 📋 **Comment on travaille.** Pour chaque technique : une démonstration que
> vous suivez, puis **deux exercices que vous faites** — le premier a des
> `____` à remplir, le second est une cellule vide où vous écrivez tout. La
> cellule de vérification vous dit tout de suite si votre réponse est bonne.

## 1. Le point de départ

Cette cellule est présente au début de **tous** les notebooks du cours. Elle
charge les outils dont nous aurons besoin et règle l'affichage pour les petits
écrans. Exécutez-la (bouton ▶) sans chercher à la comprendre pour l'instant.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@bf9431e/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Les trois premières lignes chargent des **bibliothèques** : du code déjà écrit
par d'autres, que vous appelez ensuite en une instruction. Le `as pd` leur
donne un **surnom** — tout le monde utilise les mêmes, et tous les exemples que
vous trouverez en ligne les emploient.

| La ligne | Ce qu'elle apporte | Quand elle sert |
|---|---|---|
| `import pandas as pd` | les **tableaux** : charger, filtrer, regrouper, calculer | dès aujourd'hui, et partout ensuite |
| `import numpy as np` | le **calcul sur des colonnes entières** : classer, remplacer, marquer des valeurs manquantes | en 2.2, pour nettoyer |
| `import matplotlib.pyplot as plt` | les **graphiques** : courbes, barres, histogrammes | en 2.3, pour visualiser |

**`pandas` est celle qui compte.** Pensez-y comme à **un Excel qu'on pilote par
des instructions** : même objet — un tableau de lignes et de colonnes — mais au
lieu de cliquer, on écrit ce qu'on veut. L'avantage : ça marche sur 45 000
lignes aussi vite que sur 10, et on peut relancer exactement la même analyse le
mois prochain.

Les deux autres attendront leur tour. Elles sont dans la cellule dès
aujourd'hui parce que **cette cellule est la même dans tous les notebooks du
cours** : vous n'aurez jamais à vous demander laquelle exécuter.

Les deux `set_option` qui suivent règlent la **largeur d'affichage** : sans
elles, un tableau de six colonnes part en accordéon sur une tablette. La
dernière ligne, `BASE`, est l'adresse web du dossier de données — c'est elle
qui vous évite de télécharger quoi que ce soit.

## 2. Charger les données

Une seule ligne. `pd.read_csv()` accepte directement une **adresse web** :
rien à télécharger, rien à ranger dans un dossier.

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")  ## lit le fichier en ligne
ventes.head(3)                             ## les 3 premieres lignes

`ventes` est un **DataFrame** : le nom que pandas donne à un tableau.

`.head(3)` affiche les 3 premières lignes. Toujours commencer par là : c'est
la façon la plus rapide de vérifier que le fichier est bien celui qu'on croit.

> 💡 `head()` prend en argument le **nombre de lignes** à afficher :
> `head(3)` en montre trois, `head(20)` en montre vingt. Sans argument, elle
> en affiche cinq.

## 3. La carte d'identité d'un fichier

Deux commandes, tout de suite après le chargement. En 30 secondes vous savez
à quoi vous avez affaire.

In [ ]:
ventes.shape   ## (nombre de lignes, nombre de colonnes)

In [ ]:
ventes.info()  ## les colonnes, leur type, les valeurs manquantes

Ce que `info()` vous dit, ligne par ligne :

- **`45123 entries`** — 45 123 lignes.
- **`non-null`** — combien de valeurs sont renseignées. Ici tout est complet ;
  on verra en séance 2.2 que c'est très rare dans la vraie vie.
- **`Dtype`** — le *type* de chaque colonne, et c'est le plus important :
  - `int64` : nombre entier (`qte`, `client_id`)
  - `float64` : nombre à virgule (`prix`)
  - `object` : **du texte** (`date`, `prod_id`)

> ⚠️ Regardez `date` : son type est `object`, c'est-à-dire du **texte**. Pour
> pandas, `"2011-10-04"` est une chaîne de caractères, pas une date. On ne peut
> donc pas encore lui demander « quel mois ? ». On corrigera ça en séance 2.2.

In [ ]:
# 45 123 lignes, mais combien de clients et de commandes distincts ?
print("clients   :", ventes["client_id"].nunique())   ## valeurs distinctes
print("commandes :", ventes["cmd_id"].nunique())      ## idem sur la commande

**Une ligne n'est pas un client.** Une ligne est *un produit dans une
commande*. 45 123 lignes correspondent à 1 955 commandes passées par 472
clients.

C'est la première question à se poser devant n'importe quel fichier :
**une ligne, c'est quoi exactement ?** Se tromper là-dessus, c'est se tromper
sur tout le reste de l'analyse.

---

### ✏️ Exercice — Le catalogue produits

> **Votre mission :**
> - Charger `produits.csv` → `produits`, puis afficher ses 3 premières lignes.
> - Combien de références contient le catalogue ? → `nb_produits`
> - *Rappel :* `df.shape` renvoie `(lignes, colonnes)` ; `shape[0]` est le nombre de lignes.

In [ ]:
produits = pd.read_csv(BASE + "produits.csv")   ## meme commande, autre fichier
nb_produits = produits.shape[0]   ## [0] = les lignes, [1] = les colonnes

print(nb_produits, "references au catalogue")
produits.head(3)

In [ ]:
verifier("references au catalogue", nb_produits == 2956,
         "shape[0] donne les lignes, shape[1] les colonnes")

---

### ✏️ Exercice — Les deux bouts du fichier

> **Votre mission :**
> - Afficher les **25 premières** lignes de `ventes`, puis les **10 dernières**.
> - Mettre la valeur de `date` de la toute dernière ligne dans `derniere_date`.
> - Le fichier couvre-t-il des périodes comparables d'un bout à l'autre ? Regardez le mois de la dernière ligne.
> - *Nouveau :* `df.tail(n)` fait ce que `head(n)` fait, mais par la fin.

In [ ]:
print(ventes.head(25))   ## le debut : ce qu'on regarde toujours

fin = ventes.tail(10)   ## la fin : ce qu'on oublie presque toujours
derniere_date = fin["date"].iloc[-1]   ## la toute derniere ligne
print(derniere_date)
fin

In [ ]:
verifier("derniere date du fichier", derniere_date.startswith("2011-12-09"),
         "tail(10) puis la colonne date de la derniere ligne")

Un fichier est presque toujours trié par date. Regarder la fin, c'est vérifier
qu'il ne s'arrête pas **au milieu d'une période** : ici il s'arrête le
9 décembre. Décembre est donc incomplet dans ce fichier — retenez-le, ça
comptera en séance 2.3, quand on le tracera.

## 4. Choisir ce qu'on regarde

### Une colonne

In [ ]:
ventes["prix"].head(3)   ## une seule colonne : c'est une Series

### Plusieurs colonnes

Doubles crochets : les crochets extérieurs veulent dire « je sélectionne »,
les intérieurs délimitent la **liste** des colonnes voulues.

In [ ]:
ventes[["prod_id", "qte", "prix"]].head(3)  ## une liste -> un tableau

### Naviguer grâce aux lignes du Dataset— `.loc`

`.loc` raisonne en **étiquettes** : le nom de la ligne, le nom de la colonne.
Ici les lignes n'ont pas reçu de nom, alors pandas leur a donné leur numéro
d'arrivée. C'est pour ça que `.loc[0]` et `.iloc[0]` renvoient la même ligne —
mais **c'est une coïncidence**, pas une règle, et elle ne survit pas au
premier filtrage.

In [ ]:
ventes.loc[10, "prix"]   ## ligne d'etiquette 10, colonne nommee "prix"

In [ ]:
# Avec .loc on nomme les colonnes au lieu de les compter
ventes.loc[0:2, ["qte", "prix"]]   ## et la borne 2 est INCLUSE

---

### ✏️ Exercice — Sélectionner avec `.loc`

> **Votre mission :**
> - Afficher toutes les lignes des colonnes **`cmd_id`**, **`prod_id`** et **`qte`** de `ventes` → `colonnes`.
> - Afficher les lignes dont la quantité **`qte` est supérieure à 2**, en conservant uniquement les colonnes **`prod_id`**, **`qte`** et **`prix`** → `ventes_filtrees`.
> - Mettre le nombre de lignes de `ventes_filtrees` dans `n_ventes`.
> - *Nouveau :* avec `.loc`, on écrit `df.loc[lignes, colonnes]` — les lignes avant la virgule, les colonnes après. Un `:` seul signifie « toutes les lignes ».

In [ ]:
colonnes = ventes.loc[:, ["cmd_id", "prod_id", "qte"]]      ## : = toutes les lignes
ventes_filtrees = ventes.loc[ventes["qte"] > 2,
                             ["prod_id", "qte", "prix"]]    ## condition, puis colonnes
n_ventes = len(ventes_filtrees)

print(colonnes.shape, "->", n_ventes, "lignes de plus de 2 unites")
ventes_filtrees.head(3)

In [ ]:
verifier(
    "selection des trois colonnes",
    list(colonnes.columns) == ["cmd_id", "prod_id", "qte"]
    and len(colonnes) == len(ventes),
    '.loc[:, ["cmd_id", "prod_id", "qte"]] : avant la virgule les lignes, apres les colonnes'
)

verifier(
    "filtrage des ventes",
    ventes_filtrees.equals(
        ventes.loc[
            ventes["qte"] > 2,
            ["prod_id", "qte", "prix"]
        ]
    ),
    'placez la condition ventes["qte"] > 2 avant la virgule et les colonnes apres'
)

verifier(
    "nombre de ventes filtrees",
    n_ventes == len(ventes_filtrees),
    "utilisez len(ventes_filtrees)"
)

## 5. Ne garder qu'une partie des lignes — `.query()`

On veut les ventes dont le prix dépasse 50 € :

In [ ]:
cheres = ventes.query("prix > 50")   ## la condition, entre guillemets
print(cheres.shape)                  ## combien de lignes ont survecu ?
cheres.head(3)

`.query()` prend une **condition écrite entre guillemets**, presque en français :
`"prix > 50"`, `"qte >= 100"`, `"pays == 'France'"`.

Vous rencontrerez aussi l'autre écriture, plus classique :

```python
ventes[ventes["prix"] > 50]
```

Les deux font exactement la même chose. **Nous utiliserons `.query()` dans ce
cours** : deux fois moins de ponctuation à taper, et beaucoup plus lisible dès
que la condition se complique.

### Une liste de valeurs — `in`

Pour retenir plusieurs valeurs d'une même colonne, inutile d'empiler les
`or` : `in` teste l'appartenance à une liste. Chargeons le fichier des
clients pour l'essayer sur des noms de pays.

In [ ]:
clients = pd.read_csv(BASE + "clients.csv")   ## une ligne = un client

# Attention aux guillemets : doubles a l'exterieur, simples a l'interieur
sud = clients.query("pays in ['Espagne', 'Portugal', 'Italie']")   ## in
print(len(sud), "clients dans ces trois pays")

> ⚠️ **Les guillemets imbriqués.** Toute la condition est une chaîne de
> caractères : guillemets **doubles à l'extérieur**, **simples à l'intérieur**
> pour les noms de pays. C'est la seule chose qui bloque vraiment sur cette
> tournure.

### Un intervalle

Un encadrement s'écrit comme en mathématiques, d'un seul tenant.

In [ ]:
moyennes = ventes.query("50 <= qte <= 100")   ## un encadrement
print(len(moyennes), "lignes")

### Ce qu'il y a dans les crochets — une colonne de Vrai/Faux

Prenez `ventes["prix"] > 50` tout seul, sans les crochets autour. Comparer une
colonne à une valeur ne rend pas **un** résultat : pandas pose la question à
**chacune des 45 123 lignes** et rend **une réponse par ligne**, `True` ou
`False`.

In [ ]:
depasse = ventes["prix"] > 50   ## une question posee a chaque ligne

print(depasse.head(3))   ## trois reponses, et le type : bool
print(depasse.sum(), "lignes repondent True")   ## .sum() les compte

Trois lignes, trois `False`, et en bas `dtype: bool` : c'est bien une colonne
de Vrai/Faux, aussi longue que le tableau. Et **41** — exactement le nombre de
lignes que `.query("prix > 50")` a gardées au début de cette section. C'est le
même objet : une fois pour compter, une fois pour filtrer.

Deux opérations se branchent dessus, et vous les retrouverez jusqu'au bloc 4 :

| Vous écrivez | Vous obtenez |
|---|---|
| `(ventes["prix"] > 50).sum()` | le **nombre** de lignes qui répondent `True` |
| `(ventes["prix"] > 50).mean()` | leur **part**, entre 0 et 1 |

> 💡 `.sum()` compte les `True` parce qu'un `True` vaut 1 et un `False` 0.
> C'est la tournure la plus fréquente du cours : « combien de lignes remplissent
> cette condition ? » s'écrit `(condition).sum()`.

---

### ✏️ Exercice — Les grosses commandes

> **Votre mission :**
> - Ne garder que les lignes de **plus de 100 unités** → `grosses`.
> - Combien y en a-t-il ? → `nb_grosses`

In [ ]:
grosses = ventes.query("qte > 100")   ## la condition entre guillemets
nb_grosses = grosses.shape[0]         ## shape[0] = le nombre de lignes

print(nb_grosses, "lignes de plus de 100 unites")

In [ ]:
verifier("grosses commandes", nb_grosses == 560,
         "query() prend la condition entre guillemets : \"qte > 100\"")

---

### ✏️ Exercice — Beaucoup d'unités, petit prix

> **Votre mission :**
> - Le service achats cherche les lignes qui partent **en volume à bas prix** : entre 50 et 100 unités, à moins de 2 € l'unité.
> - Combien y en a-t-il ? → `nb_volume`
> - Tout est à écrire : un encadrement **et** une seconde condition, dans la même chaîne. *Nouveau :* on empile deux conditions avec `and`.

In [ ]:
# Un encadrement et une condition supplementaire dans la meme chaine :
# query lit "50 <= qte <= 100 and prix < 2" comme une phrase
volume = ventes.query("50 <= qte <= 100 and prix < 2")
nb_volume = len(volume)

print(nb_volume, "lignes en volume a bas prix")

In [ ]:
verifier("volume a bas prix", nb_volume == 826,
         "un encadrement 50 <= qte <= 100, puis and prix < 2, dans la meme chaine")

## 6. Résumer en un coup d'œil

### `describe()` — le résumé chiffré

In [ ]:
# On selectionne les colonnes AVANT : sinon la sortie deborde de l'ecran
ventes[["qte", "prix"]].describe().round(2)   ## huit statistiques d'un coup

À lire ainsi :

- `mean` : la moyenne. Prix moyen : **3,93 €**.
- `50%` : la **médiane**, la valeur qui coupe la population en deux. **1,95 €**.
- `max` : la valeur maximale. **4 161 €**.

> 📊 Moyenne 3,93 € mais médiane 1,95 € : la moyenne est **deux fois** la
> médiane. C'est la signature d'une poignée de valeurs très élevées qui tirent
> la moyenne vers le haut. Devant un écart pareil, la médiane décrit bien mieux
> « le produit typique ». Un réflexe à garder : **comparer moyenne et médiane
> avant de citer un chiffre en réunion.**

### `value_counts()` — compter les catégories

In [ ]:
clients["segment"].value_counts()   ## deja trie du plus frequent

### Un seul chiffre à la fois

In [ ]:
print("prix moyen :", ventes["prix"].mean().round(2))   ## .round() arrondit
print("quantite max :", ventes["qte"].max())            ## le maximum
print("lignes a plus de 50 euros :", cheres.shape[0])   ## shape[0] = lignes

---

### ✏️ Exercice — Le pays le plus représenté

> **Votre mission :**
> - Compter les clients par pays → `par_pays`.
> - Mettre le nom du pays le plus représenté dans `pays_top` et son nombre de clients dans `nb_top`.
> - *Indice :* `value_counts()` trie déjà du plus fréquent au moins fréquent.

In [ ]:
par_pays = clients["pays"].value_counts()   ## deja trie

# .index donne les etiquettes, .iloc donne les valeurs par position
pays_top = par_pays.index[0]   ## le pays en tete
nb_top = par_pays.iloc[0]      ## son effectif

print(pays_top, ":", nb_top, "clients")

In [ ]:
verifier("pays le plus represente", pays_top == "Royaume-Uni",
         "value_counts() est deja trie : le premier est le plus frequent")
verifier("nombre de clients", nb_top == 235,
         ".index[0] donne l'etiquette, .iloc[0] donne l'effectif")

---

### ✏️ Exercice — La répartition, en pourcentage

> **Votre mission :**
> - Quelle **part** des clients chaque pays représente-t-il ? En %, arrondi à 1 décimale.
> - Afficher les cinq premiers, et mettre la part du Royaume-Uni dans `part_uk`.
> - *Nouveau :* `value_counts(normalize=True)` renvoie des parts (entre 0 et 1) au lieu d'effectifs.

In [ ]:
part = (clients["pays"].value_counts(normalize=True) * 100).round(1)   ## en %

part_uk = part["Royaume-Uni"]
print(part.head(5))

# Le Royaume-Uni pese la moitie du fichier clients a lui seul : c'est le
# marche domestique. On verra en 2.3 que sa part du CHIFFRE D'AFFAIRES
# n'est pas la meme que sa part des clients.

In [ ]:
verifier("part du Royaume-Uni", part_uk == 49.8,
         "normalize=True donne une part entre 0 et 1 : multipliez par 100")

## 7. Calculer sur des colonnes entières

Le chiffre d'affaires d'une ligne, c'est la quantité multipliée par le prix.

In [ ]:
ventes["ca"] = ventes["qte"] * ventes["prix"]  ## 45 123 calculs
ventes[["qte", "prix", "ca"]].head(3)          ## toujours verifier apres

Regardez bien ce qui vient de se passer : **une seule instruction a fait
45 123 multiplications**. On écrit l'opération *une fois, sur la colonne*, et
pandas l'applique à chaque ligne. C'est ce qui rend l'analyse de 45 000 lignes
aussi simple que celle de 10.

### Les opérations sur les nombres

Toutes les opérations arithmétiques fonctionnent de cette façon :

| Vous écrivez | Ce que pandas fait |
|---|---|
| `df["a"] + df["b"]` | additionne les deux colonnes, ligne à ligne |
| `df["a"] - 10` | retire 10 à **chaque** ligne |
| `df["a"] * 1.2` | multiplie chaque ligne par 1,2 |
| `df["a"] / df["a"].sum()` | divise chaque ligne par le total |
| `df["a"] ** 2` | met chaque ligne au carré |
| `df["a"].round(2)` | arrondit chaque ligne à 2 décimales |

In [ ]:
ventes["ttc"] = (ventes["prix"] * 1.2).round(2)   ## TVA a 20 %

ventes[["prix", "ttc", "qte", "ca"]].head(3)

Deux formes différentes cohabitent ici, et il faut les distinguer :

- `ventes["prix"] * 1.2` applique **le même nombre** à toutes les lignes ;
- `ventes["qte"] * ventes["prix"]` fait travailler **deux colonnes ensemble**,
  ligne par ligne.

### Et si on fait la même chose sur du texte ?

Les mêmes signes existent, mais ils ne veulent pas dire la même chose. Vous
l'aviez déjà croisé au bloc 1 avec `2 * "3"`.

In [ ]:
pays = clients["pays"]   ## une colonne de texte

print((pays + " (Europe ?)").head(2).tolist())   ## le + COLLE deux textes
print((pays * 2).head(2).tolist())               ## l'etoile REPETE le texte

Aucune erreur, et pourtant rien de ce qu'on attendait d'un `+` ou d'un `*`.

Le vrai danger n'est pas là : il est dans une colonne de **nombres stockés en
texte**. Le calcul « marche », et le résultat est absurde.

In [ ]:
# Comme le ferait un export mal configure : des quantites en texte
qte_txt = ventes["qte"].head(5).astype(str)

print("somme des nombres :", ventes["qte"].head(5).sum())
print("somme des textes  :", qte_txt.sum())   ## colle bout a bout

`96` d'un côté, `2424121224` de l'autre, et **aucun message d'erreur**. C'est
précisément ce qui vous attend en séance 2.2, où le prix arrive écrit
`2,08 EUR`. D'où le réflexe : après un `read_csv`, regarder les types avec
`info()` **avant** de calculer quoi que ce soit.

### Travailler volontairement sur du texte — `.str`

Pour manipuler du texte, pandas range ses outils derrière `.str`. Là encore,
toute la colonne est traitée d'un coup.

In [ ]:
print(pays.str.upper().head(2).tolist())   ## tout en majuscules
print(pays.str.len().head(3).tolist())     ## longueur de chaque chaine

# Des Vrai/Faux, comme au paragraphe 5 : .sum() les compte
print(pays.str.startswith("F").sum(), "clients dans un pays en F")

Les six commandes de texte à connaître :

| Commande | Effet |
|---|---|
| `.str.lower()` / `.str.upper()` | tout en minuscules / en majuscules |
| `.str.strip()` | enlève les espaces au début et à la fin |
| `.str.len()` | la longueur de chaque chaîne |
| `.str.replace("a", "b")` | remplace un morceau de texte |
| `.str.contains("Pays")` | vrai si la chaîne contient ce texte |
| `.str.startswith("F")` | vrai si elle commence par ce texte |

> 💡 Elles reviendront toutes en séance 2.2 : c'est avec elles qu'on répare
> une colonne de texte mal saisie.

---

### ✏️ Exercice — Le chiffre d'affaires total

> **Votre mission :**
> - Calculer le chiffre d'affaires **total** de l'année → `ca_total`, arrondi à 2 décimales.
> - La colonne `ca` existe déjà : il ne reste qu'à la sommer.

In [ ]:
# sum() additionne toute la colonne, ligne par ligne
ca_total = round(ventes["ca"].sum(), 2)

print("CA total :", ca_total, "euros")

In [ ]:
verifier("chiffre d'affaires total", ca_total == 1152913.87,
         "sum() sur la colonne ca, puis round(..., 2)")

---

### ✏️ Exercice — La plus grosse ligne du fichier

> **Votre mission :**
> - Retrouver la **ligne entière** dont le chiffre d'affaires est le plus élevé → `ligne_max`.
> - Mettre son `prod_id` dans `prod_max`, puis chercher ce `prod_id` dans `produits`.
> - Est-ce vraiment un produit ?
> - *Nouveau :* `df['ca'].idxmax()` donne l'**étiquette** de la ligne du maximum ; `df.loc[...]` va ensuite la chercher.

In [ ]:
ligne_max = ventes.loc[ventes["ca"].idxmax()]   ## idxmax = l'etiquette
prod_max = ligne_max["prod_id"]
print(ligne_max)

# prod_id vaut "M" : 4 161 EUR sur une seule ligne, et le catalogue ne
# le connait pas. "M" veut dire Manual, une saisie manuelle de facturation.
produits.query("prod_id == 'M'")

In [ ]:
verifier("la plus grosse ligne", prod_max == "M",
         "idxmax() sur la colonne ca, puis .loc pour aller chercher la ligne")

In [ ]:
ventes["Prix"]   ## erreur volontaire : la colonne est "prix"

In [ ]:
# Le reflexe quand on ne se souvient plus d'un nom de colonne
ventes.columns   ## la liste exacte, majuscules comprises

## 8. Nettoyer une base de données

`ventes.csv` était **impeccable** : pas un trou, pas un doublon, des types
corrects. Ça n'arrive jamais.

Voici le même détaillant, mais l'export tel qu'il sort vraiment du système :
`ventes_sale.csv`.

> 🎯 **Votre mission désormais:** transformer le fichier de données sales en données
> exploitables, et savoir dire **combien de lignes** vous avez perdues au
> passage et **pourquoi**.

> 📋 **Comment on travaille.** Cinq défauts. Pour chacun : une démonstration
> sur une table appelée `propre`, puis **deux exercices** où vous nettoyez la
> vôtre, appelée `net` — un à trous, un que vous écrivez en entier. À la fin,
> le fichier propre, c'est vous qui l'aurez construit.

In [ ]:
sale = pd.read_csv(BASE + "ventes_sale.csv")   ## le fichier brut

print(sale.shape)   ## (lignes, colonnes)
sale.head(5)        ## cinq lignes suffisent a reperer l'essentiel

Prenez 30 secondes pour regarder ces cinq lignes. Qu'est-ce qui cloche ?

In [ ]:
sale.info()   ## regarder surtout la colonne Dtype

### Le diagnostic

`info()` révèle déjà deux problèmes graves :

- **`prix` est de type `object`** — c'est du **texte**, pas un nombre. On ne
  peut donc rien calculer avec. Les coupables sont visibles dès les cinq
  premières lignes : le suffixe de `0,42 EUR`, et la **virgule** décimale de
  `3,75` là où Python attend un point.
- **`date` est de type `object`** — du texte aussi. Impossible de demander
  « quel mois ? ».

Et un troisième, visible sur le `non-null` :

- **`client_id` a des trous.**

Il y en a quatre autres qu'`info()` ne montre pas. On va les débusquer.

## Défaut 1 — Les doublons

**On commence toujours par là**, avant toute autre étape de nettoyage.

La raison est un problème de comptage. Imaginez que vous commenciez par
retirer les ventes sans client : vous notez « 407 lignes retirées ». Mais
parmi ces 407, certaines étaient des copies l'une de l'autre. Vous n'avez donc
pas retiré 407 ventes, vous en avez retiré moins — et vous ne saurez jamais
combien. En dédoublonnant d'abord, chaque ligne du fichier est une vente
distincte, et tous les comptes qui suivent veulent dire quelque chose.

`duplicated()` ne rend pas un nombre. Comme la comparaison `prix > 50` de la
séance 2.1, il pose une question **à chaque ligne** — « celle-ci, je l'ai déjà
vue plus haut ? » — et rend **une réponse `True` ou `False` par ligne**, soit
5 370 réponses ici. Pour en tirer un nombre, on compte les `True` avec
`.sum()`.

In [ ]:
marques = sale.duplicated()   ## True = ligne deja vue plus haut

print(marques.iloc[140:145])   ## la 143e est une copie d'une ligne d'avant
print("lignes strictement identiques :", marques.sum())   ## .sum() compte les True

propre = sale.drop_duplicates().copy()
print(sale.shape[0], "->", propre.shape[0], "apres suppression des doublons")

> ⚠️ Nuance importante : ici les lignes sont **strictement identiques sur
> toutes les colonnes**, donc on peut supprimer. Si seul le `cmd_id` était en
> double, ce pourrait être un vrai client commandant deux fois le même
> article. Vérifiez toujours **sur quelles colonnes** porte le doublon avant
> de supprimer.

## Défaut 2 — Les valeurs manquantes

La commande à taper devant n'importe quel fichier :

In [ ]:
propre.isna().sum()   ## un compte de trous, colonne par colonne

407 lignes sans `client_id`. **Que faire ?**

Il n'y a pas de réponse universelle. Il y a une question à se poser :
**pourquoi cette valeur manque-t-elle ?**

Ici, probablement des ventes sans compte client (achat en magasin, commande
invitée). Donc :

| Votre question | La bonne décision |
|---|---|
| « Combien mes clients dépensent-ils ? » | **Supprimer** ces lignes : elles n'ont pas de client |
| « Quel est mon chiffre d'affaires total ? » | **Les garder** : ce sont de vraies ventes, les retirer fausserait le total |

### À quoi ressemblerait `fillna`, concrètement

`fillna(valeur)` remplace chaque trou par la valeur qu'on lui donne. Sur
cette colonne, ça s'écrirait comme ceci — regardez le résultat avant de
trouver la commande pratique.

In [ ]:
# On ecrit dans une colonne A COTE, jamais par-dessus l'originale
propre["client_id_new"] = propre["client_id"].fillna(0)   ## 0 dans les trous

print("trous restants :", propre["client_id_new"].isna().sum())
print(propre["client_id_new"].value_counts().head(3))

Plus un seul trou : mission accomplie ? Regardez le classement. Le « client
0 » arrive **deuxième du fichier** avec 407 achats, derrière un seul client
réel. Sauf que ce client n'existe pas : ce sont 407 acheteurs différents
regroupés sous une étiquette inventée. Toute analyse par client sera fausse,
et absolument rien ne vous préviendra.

> ⚠️ **Le piège à ne jamais commettre :** `fillna(0)` sur un identifiant.
> Remplir une valeur manquante, c'est **inventer une donnée** — ne le faites
> que si vous pouvez le justifier.

`fillna` a pourtant des usages parfaitement légitimes : une quantité absente
qu'on sait valoir 0, un libellé vide qu'on remplace par `"inconnu"`, un prix
manquant qu'on remplace par la médiane de sa catégorie. La question n'est
jamais « est-ce que ça marche ? » mais « qu'est-ce que j'affirme en
remplissant ce trou ? ».

In [ ]:
# Celle-la ne nous sert a rien : on la retire avant de continuer
propre = propre.drop(columns=["client_id_new"])   ## drop(columns=[...])

In [ ]:
# Notre question portera sur les clients : on supprime ces lignes,
# mais on note combien on en perd.
avant = len(propre)
propre = propre.dropna(subset=["client_id"]).copy()   ## cette colonne seule

print(avant, "->", len(propre), f"({avant - len(propre)} lignes retirees)")

## Défaut 3 — Des nombres stockés en texte

C'est le défaut le plus courant, et le plus sournois.

In [ ]:
propre["prix"].head(4)   ## du texte, pas des nombres

Deux problèmes dans une seule colonne :

1. Le suffixe **` EUR`** sur certaines valeurs.
2. La **virgule** comme séparateur décimal — convention française, alors que
   Python attend un point.

Trois étapes, dans cet ordre :

In [ ]:
# .str donne acces aux operations sur du texte, colonne entiere d'un coup
prix_txt = propre["prix"].str.replace(" EUR", "")   ## 1. l'unite
prix_txt = prix_txt.str.replace(",", ".")           ## 2. la virgule

propre["prix"] = pd.to_numeric(prix_txt, errors="coerce")   ## 3. la conversion
propre["prix"].head(4)   ## le type a change : ce sont des nombres

**`errors="coerce"`** veut dire : *« si tu n'arrives pas à convertir une
valeur, mets `NaN` au lieu de tout faire planter »*. C'est très pratique —
et très dangereux si on ne vérifie pas ensuite.

In [ ]:
# Reflexe obligatoire apres un coerce : combien de valeurs ont ete perdues ?
print("prix non convertis :", propre["prix"].isna().sum())

Zéro. Notre conversion est propre. **Faites systématiquement cette
vérification** : sans elle, vous pourriez transformer silencieusement 3 000
prix en `NaN` et ne vous en apercevoir qu'en présentant vos résultats.

## Défaut 4 — Les dates

Le plus piégeux. On procède en trois temps — mais d'abord, une question de
méthode.

### Temps 0 : savoir ce qu'on attend

On ne peut pas juger si une conversion a réussi sans savoir à quoi devrait
ressembler le résultat. La colonne est encore du texte, mais du texte régulier :
`24/11/2011`, `24-11-2011`. Ses **quatre derniers caractères** sont donc
l'année, et ses **troisième et quatrième** le mois, quel que soit le séparateur.
Cela suffit à cadrer la période sans rien convertir.

In [ ]:
# .str[3:5] et .str[-4:] : on decoupe le texte par position
annee = propre["date"].str[-4:]    ## les 4 derniers caracteres
mois = propre["date"].str[3:5]     ## les 3e et 4e

print(annee.value_counts().to_dict())
print("mois presents en 2010 :", sorted(mois[annee == "2010"].unique()))

Deux années : 335 lignes en 2010 et 4 377 en 2011. Et en 2010, **un seul
mois** — décembre. Ce fichier couvre donc **de décembre 2010 à décembre
2011**, ce qui est cohérent avec l'historique d'un an qu'on nous a annoncé.

Retenez ce repère : c'est lui qui va nous permettre, dans deux cellules, de
repérer une conversion qui a échoué sans le dire.

**Temps 1 :** la façon naïve.

In [ ]:
# Cellule volontairement fausse : lisez le message d'erreur
pd.to_datetime(propre["date"])   ## sans format, pandas devine... et echoue

`ValueError: time data "24-11-2011" doesn't match format "%d/%m/%Y"`

Le fichier mélange deux écritures : `14/11/2011` et `24-11-2011`. pandas veut
un format unique. Le message suggère lui-même la solution : `format="mixed"`.

**Temps 2 :** on ajoute `format="mixed"`.

In [ ]:
essai = pd.to_datetime(propre["date"], format="mixed")   ## deux ecritures

print("date la plus ancienne :", essai.min())
print("date la plus recente  :", essai.max())

Plus d'erreur. Mais confrontez le résultat au repère du temps 0 : le fichier
va de décembre 2010 à décembre 2011, et pandas annonce **janvier 2010**.
**C'est faux.**

Pourquoi ? Parce que `01/12/2010` a été lu **à l'américaine** : mois d'abord,
donc le 12 janvier. En français, c'est le 1er décembre.

**Temps 3 :** on impose la lecture française avec `dayfirst=True`.

In [ ]:
# dayfirst=True : lecture francaise, le jour avant le mois
propre["date"] = pd.to_datetime(propre["date"], format="mixed", dayfirst=True)

print("date la plus ancienne :", propre["date"].min())
print("date la plus recente  :", propre["date"].max())

> ⚠️ **Le point le plus important de la séance.** L'étape 1 produisait une
> **erreur bruyante** : gênante, mais elle vous arrête. L'étape 2 produisait
> une **erreur silencieuse** : le code tourne, les chiffres s'affichent, et
> ils sont faux. C'est de très loin la plus dangereuse.
>
> Après toute conversion, **vérifiez que le résultat est plausible** :
> `.min()`, `.max()`, un `head()`. Trente secondes qui vous éviteront de
> présenter des chiffres faux.

Une fois la colonne convertie en date, `.dt` ouvre tout :

In [ ]:
propre["mois"] = propre["date"].dt.month           ## .dt = boite a outils
propre["jour_sem"] = propre["date"].dt.dayofweek   ## 0 = lundi, 6 = dimanche

propre[["date", "mois", "jour_sem"]].head(3)

## Défaut 5 — Du texte incohérent

In [ ]:
print("nombre de categories distinctes :", propre["categorie"].nunique())
propre["categorie"].unique()[:8]

24 catégories, alors qu'il n'en existe que 8. Regardez bien : `' cuisine'`
avec un espace devant, `'CUISINE'` en majuscules, `'cuisine'`. Pour pandas,
ce sont **trois catégories différentes** — et un `value_counts()` sur cette
colonne éclaterait la cuisine en trois lignes, chacune sous-estimée.

> ⚠️ L'espace en début de chaîne est **invisible à l'écran**. C'est ce qui
> rend ce défaut particulièrement traître.

In [ ]:
# .str.strip() enleve les espaces au bord, .str.lower() met en minuscules
propre["categorie"] = propre["categorie"].str.strip().str.lower()   ## 24 -> 8

print("apres nettoyage :", propre["categorie"].nunique(), "categories")

## Défaut 6 — Les valeurs aberrantes

In [ ]:
propre["qte"].describe().round(1)   ## regarder min et max avant tout

Un minimum **négatif** et un maximum à **99 999**. Deux anomalies, mais elles
n'ont rien à voir :

- **`qte` négatif** : ce sont des **retours**. Ce n'est pas une erreur, c'est
  une information métier. On les écarte du calcul de chiffre d'affaires, mais
  on ne les jette pas — un taux de retour, ça s'analyse.
- **`qte = 99999`** : personne ne commande 99 999 articles. C'est une saisie
  erronée, ou un code sentinelle. On l'écarte.

> Traiter ces deux cas de la même façon serait une faute d'analyse.

In [ ]:
retours = propre.query("qte < 0")   ## un retour, pas une erreur
print("retours :", len(retours), "lignes")
print("quantites aberrantes :", len(propre.query("qte >= 10000")), "lignes")

avant = len(propre)
propre = propre.query("qte > 0 and qte < 10000").copy()   ## "and" dans query
print(avant, "->", len(propre))

## Enrichir — classer des lignes selon une règle

Une dernière commande, qui ne répare rien : elle **ajoute** une colonne. On
écrit la règle **une fois**, pandas l'applique à chaque ligne.

In [ ]:
propre["ca"] = propre["qte"] * propre["prix"]   ## calculable enfin

# np.where(condition, valeur_si_vrai, valeur_si_faux)
propre["type"] = np.where(propre["ca"] > 50, "grosse", "petite")   ## deux cas
propre["type"].value_counts()

> 💡 `np.where` ne tranche qu'entre **deux** possibilités. Pour trois catégories
> ou plus, et pour découper une colonne en tranches, deux commandes vous
> attendent dans la **feuille facultative** : `np.select` et `pd.cut`. Elles
> resserviront au bloc 3.

## Le bilan

Bonne pratique pour finir : un petit compte rendu de ce qu'on a retiré. Ça
tient en trois lignes, et ça évite d'avoir à se demander, trois semaines plus
tard, d'où viennent les lignes qui manquent.

---

## Ce que vous savez faire maintenant

| Compétence | Vous voulez... | La commande |
|---|---|---|
| Charger | charger un fichier | `pd.read_csv(BASE + "ventes.csv")` |
| Explorer | voir les premières lignes | `df.head(3)` |
| Explorer | voir les dernières lignes | `df.tail(10)` |
| Explorer | connaître la taille du tableau | `df.shape` |
| Explorer | voir les colonnes et leurs types | `df.info()` |
| Explorer | lire la liste exacte des colonnes | `df.columns` |
| Explorer | compter les valeurs distinctes | `df["col"].nunique()` |
| Sélectionner | sélectionner une colonne | `df["prix"]` |
| Sélectionner | sélectionner plusieurs colonnes | `df[["qte", "prix"]]` |
| Sélectionner | sélectionner une ligne par sa position | `df.iloc[0]` |
| Sélectionner | sélectionner une case par étiquette | `df.loc[10, "prix"]` |
| Filtrer | filtrer des lignes | `df.query("prix > 50")` |
| Filtrer | combiner deux conditions | `df.query("qte >= 100 and prix > 2")` |
| Filtrer | filtrer sur une liste de valeurs | `df.query("pays in ['France', 'Belgique']")` |
| Filtrer | filtrer sur un intervalle | `df.query("50 <= qte <= 100")` |
| Filtrer | obtenir une réponse Vrai/Faux par ligne | `df["prix"] > 50` |
| Filtrer | compter les lignes qui remplissent une condition | `(df["prix"] > 50).sum()` |
| Résumer | obtenir le résumé chiffré | `df[["qte", "prix"]].describe()` |
| Résumer | compter les catégories | `df["pays"].value_counts()` |
| Résumer | calculer un indicateur | `df["prix"].mean()`, `.median()`, `.max()`, `.sum()` |
| Calculer | créer une colonne calculée | `df["ca"] = df["qte"] * df["prix"]` |
| Nettoyer | repérer les valeurs manquantes | `df.isna().sum()` |
| Nettoyer | supprimer les lignes incomplètes | `df.dropna(subset=["client_id"])` |
| Nettoyer | remplacer les valeurs manquantes | `df["prix"].fillna(0)` |
| Nettoyer | compter les doublons | `df.duplicated().sum()` |
| Nettoyer | supprimer les doublons | `df.drop_duplicates()` |
| Nettoyer | convertir du texte en nombre | `pd.to_numeric(col, errors="coerce")` |
| Nettoyer | convertir du texte en date | `pd.to_datetime(col, format="mixed", dayfirst=True)` |
| Nettoyer | extraire le mois d'une date | `df["date"].dt.month` |
| Nettoyer | nettoyer du texte | `col.str.strip().str.lower()` |
| Nettoyer | enlever un morceau de texte | `col.str.replace(" EUR", "")` |
| Enrichir | classer selon une condition | `np.where(cond, "oui", "non")` |

Deux commandes vont plus loin et vous attendent dans la **feuille facultative** :
`np.select` pour classer selon plusieurs conditions, et `pd.cut` pour découper
une colonne de nombres en tranches.

## Les réflexes à emporter

Devant un fichier inconnu, toujours dans cet ordre : **`shape`, `info()`,
`head(3)`, `describe()`**. Trente secondes, et vous savez de quoi vous parlez.
Et la première question à se poser : **une ligne, c'est quoi exactement ?**

**Le nettoyage est un pipeline, pas une série de bricolages.** Écrivez-le dans
l'ordre, de haut en bas, en repartant toujours du fichier brut. Le jour où on
vous livre le fichier du mois suivant, vous relancez le notebook et c'est fini.

Et **notez toujours combien de lignes vous perdez à chaque étape**. Un
nettoyage qui fait disparaître 40 % des données n'est pas un nettoyage, c'est
une erreur.